# DETR：Detection Transformer

这个 Notebook 使用预训练 `facebook/detr-resnet-50` 展示 DETR（End-to-End Object Detection with Transformers，Carion et al. 2020）的推理流程与架构解读。

DETR 是第一个完全用 Transformer 做端到端目标检测的方法，摒弃了 Anchor 和 NMS。

内容包括：
- 预训练模型推理与边界框可视化
- CNN Backbone 特征图分析
- Encoder token 序列分析
- Object Query 机制解读
- Decoder Cross-Attention 可视化
- DETR vs YOLO 架构对比

## 1. 环境准备

```bash
pip install transformers pillow requests timm
```

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import requests
from io import BytesIO

plt.style.use('seaborn-v0_8')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
from transformers import DetrImageProcessor, DetrForObjectDetection

# 加载预训练模型（首次运行会自动下载权重）
processor = DetrImageProcessor.from_pretrained('facebook/detr-resnet-50')
model     = DetrForObjectDetection.from_pretrained('facebook/detr-resnet-50')
model     = model.to(device)
model.eval()
print('DETR-R50 loaded.')
print(f'Object queries: {model.config.num_queries}  (每张图最多检测 {model.config.num_queries} 个目标)')

## 2. DETR 架构概述

```
Input Image
    ↓
ResNet-50 Backbone  →  特征图 (B, 2048, H/32, W/32)
    ↓  1×1 Conv 降维
Flatten + Position Encoding  →  token 序列 (B, H/32×W/32, 256)
    ↓
Transformer Encoder (6层)  →  全局特征表示
    ↓
Transformer Decoder (6层)  ←  100 个可学习 Object Query
    ↓
FFN × 100  →  100 个 (class, box) 预测
    ↓
Hungarian Matching（训练）/ 直接过滤低置信度（推理）
```

## 3. 准备测试图像

In [ ]:
def load_image(url):
    return Image.open(BytesIO(requests.get(url, timeout=15).content)).convert('RGB')


image_urls = [
    'http://images.cocodataset.org/val2017/000000039769.jpg',  # 猫
    'http://images.cocodataset.org/val2017/000000397133.jpg',  # 街道
]

images = [load_image(u) for u in image_urls]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, img in zip(axes, images):
    ax.imshow(img)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 4. 推理与可视化

In [ ]:
COLORS = plt.cm.get_cmap('tab20').colors


def detect_and_visualize(image_pil, threshold=0.7):
    inputs  = processor(images=image_pil, return_tensors='pt')
    inputs  = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    # 把模型输出的归一化坐标转回原图像素坐标
    target_sizes = torch.tensor([image_pil.size[::-1]])
    results      = processor.post_process_object_detection(outputs, threshold=threshold,
                                                            target_sizes=target_sizes)[0]

    fig, ax = plt.subplots(1, figsize=(10, 7))
    ax.imshow(image_pil)

    for score, label, box in zip(results['scores'], results['labels'], results['boxes']):
        x1, y1, x2, y2 = box.tolist()
        cls   = model.config.id2label[label.item()]
        color = COLORS[label.item() % len(COLORS)]
        rect  = patches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                   linewidth=2, edgecolor=color, facecolor='none')
        ax.add_patch(rect)
        ax.text(x1, y1 - 4, f'{cls} {score:.2f}', color='white', fontsize=9,
                bbox=dict(facecolor=color, alpha=0.8, pad=1, edgecolor='none'))

    ax.set_title(f'DETR 检测结果（threshold={threshold}）')
    ax.axis('off')
    plt.tight_layout()
    plt.show()
    print(f'  检测到 {len(results["scores"])} 个目标')


for img in images:
    detect_and_visualize(img)

## 5. CNN Backbone 特征分析

In [ ]:
inputs    = processor(images=images[0], return_tensors='pt')
inputs    = {k: v.to(device) for k, v in inputs.items()}

# 提取 backbone 最终特征图
with torch.no_grad():
    backbone_out = model.model.backbone(inputs['pixel_values'],
                                        inputs.get('pixel_mask'))

# backbone 输出是列表，取最后一层
feat_map = backbone_out.feature_maps[-1]  # (B, 2048, H/32, W/32)
print(f'Backbone feature map : {tuple(feat_map.shape)}')
print(f'  原图大约 {feat_map.shape[2]*32}×{feat_map.shape[3]*32} → 压缩 32 倍到 {feat_map.shape[2]}×{feat_map.shape[3]}')

# 可视化前 6 个通道（用最大激活聚合）
feat_np = feat_map[0].float().cpu().numpy()
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for i, ax in enumerate(axes.flatten()):
    ax.imshow(feat_np[i], cmap='viridis')
    ax.set_title(f'Channel {i}')
    ax.axis('off')
plt.suptitle('ResNet-50 Backbone 最终特征图（前6通道）', fontsize=13)
plt.tight_layout()
plt.show()

## 6. Encoder Token 序列分析

In [ ]:
# 注册 hook 提取 encoder 输出
encoder_output = {}

def hook_encoder(module, inp, out):
    encoder_output['last'] = out.last_hidden_state.detach()


handle = model.model.encoder.register_forward_hook(hook_encoder)

with torch.no_grad():
    _ = model(**inputs)

handle.remove()

enc_out = encoder_output['last']
print(f'Encoder output shape : {tuple(enc_out.shape)}')
print(f'  (B, num_tokens, d_model) = ({enc_out.shape[0]}, {enc_out.shape[1]}, {enc_out.shape[2]})')
H = feat_map.shape[2]
W = feat_map.shape[3]
print(f'  num_tokens = {H}×{W} = {H*W}（backbone 特征图展平后的 token 数）')

## 7. Object Query 机制

In [ ]:
# Object Query 是 decoder 的输入，每个 query 负责检测一个潜在目标
queries = model.model.query_position_embeddings.weight  # (num_queries, d_model)
print(f'Object query embeddings : {tuple(queries.shape)}')
print(f'  {queries.shape[0]} 个 query，每个维度 {queries.shape[1]}，随机初始化后端到端训练')

# 可视化 query 嵌入的范数分布
norms = queries.detach().cpu().norm(dim=-1).numpy()
plt.figure(figsize=(10, 3))
plt.bar(range(len(norms)), norms)
plt.title('Object Query L2 Norm（每个 query 的嵌入强度）')
plt.xlabel('Query index')
plt.ylabel('L2 norm')
plt.tight_layout()
plt.show()

## 8. Decoder Cross-Attention 可视化

In [ ]:
# 提取最后一层 decoder 的 cross-attention 权重
attn_weights = {}

def hook_cross_attn(module, inp, out):
    # nn.MultiheadAttention 返回 (output, attn_weights)
    if isinstance(out, tuple) and len(out) == 2:
        attn_weights['weights'] = out[1].detach()  # (B, num_queries, num_tokens)


# 注册在最后一层 decoder 的 cross-attention
last_dec = model.model.decoder.layers[-1]
handle   = last_dec.encoder_attn.register_forward_hook(hook_cross_attn)

with torch.no_grad():
    outputs = model(**inputs, output_attentions=True)

handle.remove()

# 取激活最强的几个 query 对应的 attention map
target_sizes = torch.tensor([images[0].size[::-1]])
results      = processor.post_process_object_detection(outputs, threshold=0.7,
                                                        target_sizes=target_sizes)[0]

if len(results['scores']) > 0 and 'cross_attentions' in dir(outputs):
    # 使用模型自带 cross-attention 输出
    cross_attn = outputs.cross_attentions[-1]  # (B, heads, num_queries, num_tokens)
    # 对头取均值
    avg_attn   = cross_attn[0].mean(0)  # (num_queries, num_tokens)

    num_show = min(4, len(results['scores']))
    # 找得分最高的 query 对应索引
    top_queries = results['scores'].argsort(descending=True)[:num_show]

    fig, axes = plt.subplots(1, num_show, figsize=(14, 4))
    if num_show == 1:
        axes = [axes]
    for ax, qidx in zip(axes, top_queries):
        attn_map = avg_attn[qidx].reshape(H, W).cpu().numpy()
        ax.imshow(images[0], alpha=0.4)
        ax.imshow(attn_map, alpha=0.6, cmap='hot',
                  extent=[0, images[0].width, images[0].height, 0])
        lbl = model.config.id2label[results['labels'][qidx].item()]
        ax.set_title(f'Query {qidx.item()} → {lbl}')
        ax.axis('off')
    plt.suptitle('Decoder Cross-Attention Map', fontsize=13)
    plt.tight_layout()
    plt.show()
else:
    print('Cross-attention 可视化需要 output_attentions=True 且模型输出包含 cross_attentions 字段。')

## 9. 关键机制解读

### Object Query（集合预测）
- Decoder 输入 100 个可学习向量（Object Queries），每个 query 通过 Cross-Attention 询问 Encoder 输出。
- 每个 query 学到负责检测特定位置/大小目标的偏好，最终输出 100 个预测。

### 二分图匹配（Hungarian Algorithm）
- 训练时：对 100 个预测和真实框做最优一对一匹配，代价 = 分类损失 + L1 + GIoU。
- 无匹配目标的 query 被分配到「无目标」类（no-object）。
- 推理时：直接用置信度阈值过滤，无需 NMS。

### 无 NMS
- 传统方法：大量 Anchor → 预测框 → NMS 去重。DETR 用全局注意力让模型自然避免重复检测。
- 代价：训练需要更长时间（500 epoch vs YOLO 的几十 epoch）。

### Encoder 全局注意力
- Backbone 特征图展平后进入 Encoder，每个 token（空间位置）都能关注所有其他位置。
- 这使 DETR 对复杂场景中的遮挡和密集目标有独特的建模能力。

## 10. DETR vs YOLO 对比

| | DETR | YOLOv8 |
|---|---|---|
| 检测范式 | Set Prediction（集合预测） | Anchor-free 密集预测 |
| 后处理 | 无 NMS | 需要 NMS |
| 架构 | CNN Backbone + Transformer | CNN Backbone + FPN + Head |
| 小目标 | 较弱 | 较强（多尺度） |
| 训练效率 | 慢（需 500 epoch） | 快（几十 epoch） |
| 推理速度 | 较慢 | 极快（实时） |
| 可解释性 | 高（注意力图可视化） | 中 |

In [ ]:
# 参数量对比
detr_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'DETR-R50   : {detr_params:>12,} parameters')

try:
    from ultralytics import YOLO
    yolo = YOLO('yolov8n.pt')
    yolo_params = sum(p.numel() for p in yolo.model.parameters())
    print(f'YOLOv8n    : {yolo_params:>12,} parameters')
except ImportError:
    print('（安装 ultralytics 后可查看 YOLOv8 参数量）')